# Sistema de recomendación
El Movie Recommendation System es una aplicación basada en el aprendizaje automático que proporciona recomendaciones de películas personalizadas a los usuarios. Utiliza técnicas de filtrado colaborativo para analizar las preferencias del usuario y las similitudes entre las películas para generar recomendaciones precisas y relevantes. El sistema está construido utilizando el lenguaje de programación Python e incorpora bibliotecas populares de aprendizaje automático como scikit-learn y pandas.

El proyecto utiliza el conjunto de datos MovieLens, un conjunto de datos ampliamente utilizado en el campo de los sistemas de recomendación, que contiene clasificaciones de películas y metadatos. El conjunto de datos se preprocesa para crear una matriz de ítems de usuario y calcular la similitud de elementos utilizando la similitud de coseno. Esto permite que el sistema identifique películas que son similares a las que el usuario ha disfrutado anteriormente y las recomiende en consecuencia.

El proceso de recomendación implica tomar el identificador único de un usuario como entrada y generar una lista de recomendaciones de películas mejor calificadas específicamente adaptadas a sus preferencias. El sistema ajusta y actualiza dinámicamente las recomendaciones a medida que se dispone de nuevos datos.

El Sistema de Recomendación de Películas está destinado a personas que buscan sugerencias de películas personalizadas para mejorar su experiencia de ver películas. Se puede integrar en varias plataformas, como servicios de transmisión, sitios web de revisión de películas o aplicaciones de catálogo de películas personales.

In [1]:
import pandas as pd
import numpy as np


In [2]:
Movies = pd.read_csv('movies.csv')
Ratings = pd.read_csv('ratings.csv')

In [3]:
# Combinar los datos de películas y calificaciones
data = pd.merge(Ratings, Movies, on='movieId')

# Ver las primeras filas del conjunto combinado
print(data.head())

   userId  movieId  rating   timestamp                title  \
0       1      296     5.0  1147880044  Pulp Fiction (1994)   
1       3      296     5.0  1439474476  Pulp Fiction (1994)   
2       4      296     4.0  1573938898  Pulp Fiction (1994)   
3       5      296     4.0   830786155  Pulp Fiction (1994)   
4       7      296     4.0   835444730  Pulp Fiction (1994)   

                        genres  
0  Comedy|Crime|Drama|Thriller  
1  Comedy|Crime|Drama|Thriller  
2  Comedy|Crime|Drama|Thriller  
3  Comedy|Crime|Drama|Thriller  
4  Comedy|Crime|Drama|Thriller  


- userId: Identificador único del usuario que realizó la calificación.
- movieId: Identificador único de la película calificada.
- rating: Calificación otorgada por el usuario a la película, generalmente en una escala numérica (por ejemplo, 1 a 5).
- timestamp: Marca temporal del momento en que se realizó la calificación, expresada en segundos desde el inicio del tiempo Unix (01/01/1970).
- title: Título de la película.
- genres: Géneros cinematográficos asociados con la película. Los géneros están separados por barras verticales (|).

In [4]:
data

,userId,movieId,rating,timestamp,title,genres
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
1,3,296,5.0,1439474476,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2,4,296,4.0,1573938898,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
3,5,296,4.0,830786155,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
4,7,296,4.0,835444730,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
...,...,...,...,...,...,...
25000090,162358,200192,2.0,1553453039,Den frusna leoparden (1986),(no genres listed)
25000091,162358,200194,2.0,1553453843,Tough Luck (2004),Action|Adventure|Thriller
25000092,162386,139970,3.5,1549215965,I Don't Speak English (1995),Comedy
25000093,162386,200726,4.0,1554651417,The Graduates (1995),Children|Drama


SVD (Descomposición en Valores Singulares) 
Es una técnica matemática que descompone una matriz en tres componentes: U, Σ, y V^T. En sistemas de recomendación, SVD se utiliza para reducir la dimensionalidad de las matrices de calificaciones y capturar características latentes tanto de los usuarios como de los ítems.

In [6]:
# Importar la clase SVD (Singular Value Decomposition) desde la biblioteca Surprise.
# Esta clase implementa un modelo de factorización de matrices para sistemas de recomendación.
from surprise import SVD

# Instanciar un objeto del modelo SVD sin especificar parámetros adicionales.
model = SVD()

# Entrenar el modelo con el conjunto de entrenamiento (trainset).
# Este paso ajusta los parámetros del modelo para que se ajusten a las calificaciones observadas en trainset.
model.fit(trainset)

# Generar predicciones sobre el conjunto de prueba (testset) utilizando el modelo entrenado.
predictions = model.test(testset)

# Calcular y mostrar la raíz cuadrada media del error (RMSE) entre las predicciones y las calificaciones reales en testset.
import surprise.accuracy as accuracy
accuracy.rmse(predictions)


RMSE: 0.7776


0.7775894622165312

In [7]:
# Obtener todas las películas únicas
all_movies = Ratings['movieId'].unique()

# Predecir calificaciones para el usuario 1
user_predictions = []
for movie_id in all_movies:
    predicted_rating = model.predict(uid=1, iid=movie_id).est
    user_predictions.append((movie_id, predicted_rating))

# Ordenar las predicciones por calificación predicha (de mayor a menor)
user_predictions = sorted(user_predictions, key=lambda x: x[1], reverse=True)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [16]:
# Crear una matriz TF-IDF para los géneros de las películas
tfidf = TfidfVectorizer(stop_words='english')
Movies['genres'] = Movies['genres'].fillna('')  # Asegurarse de que no haya valores nulos
tfidf_matrix = tfidf.fit_transform(Movies['genres'])

# Calcular la similitud del coseno entre películas
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [19]:
# Función para obtener recomendaciones basadas en contenido
def get_recommendations(title, cosine_sim=cosine_sim):
    # Obtener el índice de la película que coincide con el título
    idx = Movies[Movies['title'] == title].index[0]
    
    # Obtener las puntuaciones de similitud de todas las películas con esa película
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Ordenar las películas según las puntuaciones de similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Obtener las puntuaciones de las 10 películas más similares (excluyendo la misma película)
    sim_scores = sim_scores[1:11]
    
    # Obtener los índices de las películas
    movie_indices = [i[0] for i in sim_scores]
    
    # Devolver los títulos de las películas más similares
    return Movies['title'].iloc[movie_indices]

In [20]:
print(get_recommendations('Toy Story (1995)'))

2203                                           Antz (1998)
3021                                    Toy Story 2 (1999)
3653        Adventures of Rocky and Bullwinkle, The (2000)
3912                      Emperor's New Groove, The (2000)
4780                                 Monsters, Inc. (2001)
9949     DuckTales: The Movie - Treasure of the Lost La...
10773                                     Wild, The (2006)
11604                               Shrek the Third (2007)
12969                       Tale of Despereaux, The (2008)
17431    Asterix and the Vikings (Astérix et les Viking...
Name: title, dtype: object
